# Condition Trades Visualisation

Loads enriched trades for a single `condition_id` and plots:
- **VWAP** + optional **Low / High** price bands (primary y-axis) at 1 min / 10 min / 1 h
- **Volume in USDC** (secondary y-axis)
- **Price spread** (max − min per bucket) in a second subplot

All prices are normalised to the **Yes** outcome (No prices are flipped to `1 − price`).
Volumes are halved to avoid double-counting the two legs of each matched trade.

Requires: `pandas`, `pyarrow`, `plotly`

In [172]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Configuration

In [173]:
# ── Data ──────────────────────────────────────────────────────────────────────
# Shard file is derived from the first hex character after '0x'.
# e.g. '0x3488...' -> enriched_3.parquet
ENRICHED_TRADES_DIR = Path("../data/trades_polygon_enriched")
CONDITION_ID        = "0x24fb7c2d95c93a68018e6c4a90d88043bb67d32fd1454924cef8ebdd550228f3"

# ── Chart ─────────────────────────────────────────────────────────────────────
# Default aggregation resolution shown on load: '1min' | '10min' | '1h'
RESOLUTION   = "1min"

# Show Low (price_min) and High (price_max) as additional traces on the price chart
SHOW_LOW     = True
SHOW_HIGH    = True

# ── Colours ───────────────────────────────────────────────────────────────────
COLORS = {
    "1min":  "#1f77b4",
    "10min": "#ff7f0e",
    "1h":    "#2ca02c",
}
VOLUME_COLOR = "rgba(150, 150, 200, 0.35)"
SPREAD_COLOR = "rgba(220, 80,  80,  0.6)"

## 1 · Load enriched trades

In [174]:
shard_char   = CONDITION_ID[2].lower()
parquet_file = ENRICHED_TRADES_DIR / f"enriched_{shard_char}.parquet"
print(f"Reading shard: {parquet_file}")

raw = pd.read_parquet(parquet_file)
raw = raw[raw["condition_id"] == CONDITION_ID]
print(f"Loaded {len(raw):,} rows for condition {CONDITION_ID}")

Reading shard: ../data/trades_polygon_enriched/enriched_2.parquet
Loaded 130,743 rows for condition 0x24fb7c2d95c93a68018e6c4a90d88043bb67d32fd1454924cef8ebdd550228f3


## 2 · Prepare trades

In [175]:
df = raw.copy()
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df.sort_values("ts", inplace=True)
df.reset_index(drop=True, inplace=True)

question = df["question"].iloc[0] if not df.empty else CONDITION_ID
slug     = df["slug"].iloc[0]     if not df.empty else ""

print(f"Question : {question}")
print(f"Slug     : {slug}")
print(f"Trades   : {len(df):,}")
print(f"Period   : {df['ts'].min()}  →  {df['ts'].max()}")
df.head(3)

Question : Will Israel launch a major ground offensive in Lebanon by March 31?
Slug     : will-israel-launch-a-major-ground-offensive-in-lebanon-by-march-31
Trades   : 130,743
Period   : 2025-11-14 06:09:20+00:00  →  2026-03-26 00:34:41+00:00


,condition_id,level_1,tx_hash,log_index,block_number,block_timestamp,trade_date,token_id,outcome,token_winner,...,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount,ts,avail_copy_qty,avail_copy_total_vol,avail_copy_count,copyable_qty
0,0x24fb7c2d95c93a68018e6c4a90d88043bb67d32fd145...,0,0xb630a4a32f8016330b31bd392004af2a0ec55bd6e95b...,589,78997831,1763100560,2025-11-14,6217461533662788881445316665765208716867293656...,Yes,True,...,BUY,Yes,6217461533662788881445316665765208716867293656...,0.23,0.298701,2025-11-14 06:09:20+00:00,0.0,0.0,0.0,0.0
1,0x24fb7c2d95c93a68018e6c4a90d88043bb67d32fd145...,1,0xb630a4a32f8016330b31bd392004af2a0ec55bd6e95b...,589,78997831,1763100560,2025-11-14,1091949505284712382216897721400313663007565546...,No,False,...,SELL,Yes,6217461533662788881445316665765208716867293656...,0.23,0.298701,2025-11-14 06:09:20+00:00,0.0,0.0,0.0,0.0
2,0x24fb7c2d95c93a68018e6c4a90d88043bb67d32fd145...,2,0xf3c58be801d1cf1567edec1ca63a731c6b93f8b5053d...,176,79017028,1763138954,2025-11-14,1091949505284712382216897721400313663007565546...,No,False,...,BUY,No,1091949505284712382216897721400313663007565546...,0.71,24.482758,2025-11-14 16:49:14+00:00,0.0,0.0,0.0,0.0


## 3 · Aggregate

In [176]:
def aggregate_trades(df: pd.DataFrame, freq: str) -> pd.DataFrame:
    """
    Resample to `freq` and return per-bar stats, all expressed as the
    Yes-outcome probability (No prices are flipped to 1 − price).

    Columns returned:
      trade_count, price_open, price_close, price_min, price_max, price_avg,
      volume_usdc, volume_shares, vwap, spread

    Volume columns are halved: each matched trade produces two rows
    (one per leg) so raw sums double-count the actual traded size.

    VWAP is computed on Yes-leg rows only (price × qty = usdc_amount
    holds exactly for Yes legs, so vwap = sum(usdc_amount) / sum(qty)).
    """
    d = df.copy()

    # Normalise all prices to the Yes outcome
    d["yes_price"] = d["price"]
    no_mask = d["outcome"].str.lower() == "no"
    d.loc[no_mask, "yes_price"] = 1.0 - d.loc[no_mask, "price"]

    ts_col = d.set_index("ts")

    agg = ts_col.resample(freq).agg(
        trade_count  =("yes_price",   "count"),
        price_open   =("yes_price",   "first"),
        price_close  =("yes_price",   "last"),
        price_min    =("yes_price",   "min"),
        price_max    =("yes_price",   "max"),
        price_avg    =("yes_price",   "mean"),
        _vol_usdc    =("usdc_amount", "sum"),
        _vol_shares  =("quantity",    "sum"),
    )

    # Halve volumes: two legs per trade
    agg["volume_usdc"]   = agg["_vol_usdc"]   / 2
    agg["volume_shares"] = agg["_vol_shares"] / 2
    agg.drop(columns=["_vol_usdc", "_vol_shares"], inplace=True)

    # VWAP from Yes legs only
    yes_only = d[d["outcome"].str.lower() == "yes"].set_index("ts")
    vwap_agg = yes_only.resample(freq).agg(
        _usd=("usdc_amount", "sum"),
        _qty=("quantity",    "sum"),
    )
    agg["vwap"] = (
        vwap_agg["_usd"] / vwap_agg["_qty"].replace(0, float("nan"))
    ).reindex(agg.index)

    # Spread: max − min Yes-price within the bucket
    agg["spread"] = agg["price_max"] - agg["price_min"]

    return agg[agg["trade_count"] > 0].copy()


RESOLUTIONS = ["1min", "10min", "1h"]

agg_data: dict[str, pd.DataFrame] = {}
for freq in RESOLUTIONS:
    agg_data[freq] = aggregate_trades(df, freq)
    print(f"{freq:>5}: {len(agg_data[freq]):,} bars")

 1min: 16,251 bars
10min: 4,006 bars
   1h: 1,076 bars


## 4 · Plot

In [177]:
def build_tooltip(agg: pd.DataFrame) -> list[str]:
    """Rich HTML tooltip for each bar (shown on the VWAP trace)."""
    tips = []
    for ts, row in agg.iterrows():
        tip = (
            f"<b>{ts.strftime('%Y-%m-%d %H:%M UTC')}</b><br>"
            f"VWAP : <b>{row['vwap']:.4f}</b><br>"
            f"Open : {row['price_open']:.4f}   Close: {row['price_close']:.4f}<br>"
            f"Low  : {row['price_min']:.4f}   High : {row['price_max']:.4f}<br>"
            f"Spread: {row['spread']:.4f}<br>"
            f"Avg  : {row['price_avg']:.4f}<br>"
            f"Vol  : <b>{row['volume_usdc']:,.2f} USDC</b>   "
            f"({row['volume_shares']:,.2f} shares)<br>"
            f"Trades: {int(row['trade_count']):,}"
        )
        tips.append(tip)
    return tips


def plot_condition(
    agg_data:          dict[str, pd.DataFrame],
    active_resolution: str,
    title:             str,
    show_low:          bool = True,
    show_high:         bool = True,
) -> go.Figure:
    """
    Two-row figure:
      Row 1: VWAP line + optional Low/High bands (primary y) and Volume bars (secondary y)
      Row 2: Spread (max − min) bar chart

    Resolution buttons toggle all traces simultaneously.
    """
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.06,
        specs=[
            [{"secondary_y": True}],
            [{"secondary_y": False}],
        ],
    )

    # traces_per_res depends on which optional traces are active
    # order per resolution: vwap, [low], [high], volume, spread
    n_optional = int(show_low) + int(show_high)
    traces_per_res = 3 + n_optional  # vwap + volume + spread + optionals

    for res_label, agg in agg_data.items():
        visible = (res_label == active_resolution)
        color   = COLORS[res_label]
        tips    = build_tooltip(agg)

        # ── VWAP ──────────────────────────────────────────────────────────────
        fig.add_trace(
            go.Scatter(
                x=agg.index, y=agg["vwap"],
                mode="lines",
                name=f"VWAP ({res_label})",
                line=dict(color=color, width=2),
                hovertext=tips, hoverinfo="text",
                visible=visible,
                legendgroup=res_label,
            ),
            row=1, col=1, secondary_y=False,
        )

        # ── Low ───────────────────────────────────────────────────────────────
        if show_low:
            fig.add_trace(
                go.Scatter(
                    x=agg.index, y=agg["price_min"],
                    mode="lines",
                    name=f"Low ({res_label})",
                    line=dict(color=color, width=1, dash="dot"),
                    hovertemplate="Low: %{y:.4f}<extra></extra>",
                    visible=visible,
                    legendgroup=res_label,
                    opacity=0.6,
                ),
                row=1, col=1, secondary_y=False,
            )

        # ── High ──────────────────────────────────────────────────────────────
        if show_high:
            fig.add_trace(
                go.Scatter(
                    x=agg.index, y=agg["price_max"],
                    mode="lines",
                    name=f"High ({res_label})",
                    line=dict(color=color, width=1, dash="dash"),
                    hovertemplate="High: %{y:.4f}<extra></extra>",
                    visible=visible,
                    legendgroup=res_label,
                    opacity=0.6,
                ),
                row=1, col=1, secondary_y=False,
            )

        # ── Volume bars ───────────────────────────────────────────────────────
        fig.add_trace(
            go.Bar(
                x=agg.index, y=agg["volume_usdc"],
                name=f"Volume ({res_label})",
                marker_color=VOLUME_COLOR,
                hovertemplate="Vol: %{y:,.2f} USDC<extra></extra>",
                visible=visible,
                legendgroup=res_label,
                showlegend=False,
            ),
            row=1, col=1, secondary_y=True,
        )

        # ── Spread bars ───────────────────────────────────────────────────────
        fig.add_trace(
            go.Bar(
                x=agg.index, y=agg["spread"],
                name=f"Spread ({res_label})",
                marker_color=SPREAD_COLOR,
                hovertemplate="Spread: %{y:.4f}<extra></extra>",
                visible=visible,
                legendgroup=res_label,
                showlegend=False,
            ),
            row=2, col=1,
        )

    # ── Resolution toggle buttons ─────────────────────────────────────────────
    n_res = len(agg_data)
    buttons = []
    for i, res in enumerate(agg_data.keys()):
        vis = [False] * (n_res * traces_per_res)
        for t in range(traces_per_res):
            vis[i * traces_per_res + t] = True
        buttons.append(dict(
            label=res,
            method="update",
            args=[{"visible": vis}, {"title": f"{title} [{res}]"}],
        ))

    fig.update_layout(
        title=f"{title} [{active_resolution}]",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", y=1.06),
        updatemenus=[dict(
            type="buttons", direction="right",
            x=0.0, y=1.13,
            showactive=True,
            buttons=buttons,
        )],
        margin=dict(t=130),
    )

    fig.update_yaxes(title_text="Yes probability",  row=1, col=1, secondary_y=False, range=[0, 1])
    fig.update_yaxes(title_text="Volume (USDC)",    row=1, col=1, secondary_y=True,  showgrid=False)
    fig.update_yaxes(title_text="Spread (max−min)", row=2, col=1)
    fig.update_xaxes(title_text="Time (UTC)",       row=2, col=1)

    return fig


title = f"{question}<br><sup>{CONDITION_ID}</sup>"
fig = plot_condition(agg_data, RESOLUTION, title, show_low=SHOW_LOW, show_high=SHOW_HIGH)
fig.show(renderer="browser")